# 6.2 Ekstraksi Fitur & Training Model (Jawa Timur, 6 Kelas)

Input: `jatim/features/features_class1..6.csv` (mean 6 band Sentinel-2A per poligon, dari `download_raster_jatim.py --phase features`).
Alur: hitung 6 indeks spektral → 12 fitur → split stratified 80/20 → banding 4 model → pilih akurasi tertinggi → simpan model + metrik.

In [1]:
import glob
import os
import numpy as np
import pandas as pd

SEED = 42
FEAT_DIR = "jatim/features"
MODEL_DIR = "jatim/model"
IMG_DIR = "jatim/gambar"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
NAMES = {1: "Sawah", 2: "Bangunan", 3: "Mangrove", 4: "Lahan hijau", 5: "Laut", 6: "Danau"}

files = sorted(glob.glob(os.path.join(FEAT_DIR, "features_class*.csv")))
files = [f for f in files if "_p" not in os.path.basename(f)]
if len(files) < 6:
    print(f"STATUS: TUNGGU — baru {len(files)}/6 CSV fitur. Jalankan download_raster_jatim.py --phase features.")
    df = None
else:
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print("baris:", len(df), "| kolom:", list(df.columns))
    print(df["class_id"].value_counts().sort_index().to_dict())

baris: 2658 | kolom: ['feature_index', 'B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'class_id']
{1: 500, 2: 500, 3: 405, 4: 471, 5: 500, 6: 282}


## Indeks spektral (rumus di 6.1) → 12 fitur

In [2]:
if df is None:
    print("STATUS: TUNGGU — lewati.")
    X = y = None
else:
    b = {c: pd.to_numeric(df[c], errors="coerce") for c in BANDS}
    B02, B03, B04, B08, B11, B12 = (b[c] for c in BANDS)
    df["NDVI"] = (B08 - B04) / (B08 + B04)
    df["NDWI"] = (B03 - B08) / (B03 + B08)
    df["NDBI"] = (B11 - B08) / (B11 + B08)
    df["NDMI"] = (B08 - B11) / (B08 + B11)
    df["EVI"] = 2.5 * (B08 - B04) / (B08 + 6 * B04 - 7.5 * B02 + 1)
    df["SAVI"] = ((B08 - B04) / (B08 + B04 + 0.5)) * 1.5
    FEATS = BANDS + ["NDVI", "NDWI", "NDBI", "NDMI", "EVI", "SAVI"]
    df = df.dropna(subset=FEATS).reset_index(drop=True)
    X = df[FEATS].to_numpy()
    y = df["class_id"].to_numpy().astype(int)
    print("fitur:", FEATS, "| sampel bersih:", len(df))
    df.to_csv(os.path.join(MODEL_DIR, "jatim_dataset_12fitur.csv"), index=False)

fitur: ['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'NDMI', 'EVI', 'SAVI'] | sampel bersih: 2658


## Banding model → akurasi tertinggi menang

Kandidat: RandomForest, SVM (RBF), KNN, HistGradientBoosting. Split stratified 80/20 (seed 42) + CV 5-fold. Metrik: accuracy, precision/recall/F1 macro.

In [3]:
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

if X is None:
    print("STATUS: TUNGGU — training dilewati.")
else:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    print("train:", len(Xtr), "test:", len(Xte))
    cands = {
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        "SVM-RBF-C10": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, random_state=SEED)),
        "KNN-15": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        "HistGradBoost": HistGradientBoostingClassifier(random_state=SEED),
    }
    rows, fitted = [], {}
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for nama, clf in cands.items():
        clf.fit(Xtr, ytr)
        p = clf.predict(Xte)
        s = cross_val_score(clf, X, y, cv=cv)
        rows.append({"model": nama, "accuracy": accuracy_score(yte, p),
                     "precision_macro": precision_score(yte, p, average="macro", zero_division=0),
                     "recall_macro": recall_score(yte, p, average="macro", zero_division=0),
                     "f1_macro": f1_score(yte, p, average="macro", zero_division=0),
                     "cv5_mean": float(s.mean())})
        fitted[nama] = clf
        print(nama, "->", {k: round(v, 4) for k, v in rows[-1].items() if k != "model"})
    res = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
    print("\n", res.round(4).to_string(index=False))
    best = res.loc[0, "model"]
    print("\nMODEL TERBAIK:", best)
    joblib.dump(fitted[best], os.path.join(MODEL_DIR, "jatim_best.joblib"))
    res.to_csv(os.path.join(MODEL_DIR, "jatim_model_compare.csv"), index=False)
    pb = fitted[best].predict(Xte)
    print(confusion_matrix(yte, pb))
    print(classification_report(yte, pb, target_names=[NAMES[i] for i in sorted(NAMES)], zero_division=0))

train: 2126 test: 532


RandomForest -> {'accuracy': 0.6541, 'precision_macro': 0.6592, 'recall_macro': 0.6498, 'f1_macro': 0.6535, 'cv5_mean': 0.6663}


SVM-RBF-C10 -> {'accuracy': 0.6767, 'precision_macro': 0.6767, 'recall_macro': 0.6694, 'f1_macro': 0.6676, 'cv5_mean': 0.6994}


KNN-15 -> {'accuracy': 0.6335, 'precision_macro': 0.6404, 'recall_macro': 0.6308, 'f1_macro': 0.6298, 'cv5_mean': 0.6437}


HistGradBoost -> {'accuracy': 0.641, 'precision_macro': 0.6492, 'recall_macro': 0.6361, 'f1_macro': 0.6418, 'cv5_mean': 0.6625}

         model  accuracy  precision_macro  recall_macro  f1_macro  cv5_mean
  SVM-RBF-C10    0.6767           0.6767        0.6694    0.6676    0.6994
 RandomForest    0.6541           0.6592        0.6498    0.6535    0.6663
HistGradBoost    0.6410           0.6492        0.6361    0.6418    0.6625
       KNN-15    0.6335           0.6404        0.6308    0.6298    0.6437

MODEL TERBAIK: SVM-RBF-C10


[[ 65   7   0  16   0  12]
 [ 20  68   0  12   0   0]
 [  7   2  67   0   0   5]
 [ 35  19   5  28   0   7]
 [  0   0   0   0 100   0]
 [ 13   1   4   6   1  32]]
              precision    recall  f1-score   support

       Sawah       0.46      0.65      0.54       100
    Bangunan       0.70      0.68      0.69       100
    Mangrove       0.88      0.83      0.85        81
 Lahan hijau       0.45      0.30      0.36        94
        Laut       0.99      1.00      1.00       100
       Danau       0.57      0.56      0.57        57

    accuracy                           0.68       532
   macro avg       0.68      0.67      0.67       532
weighted avg       0.68      0.68      0.67       532



## Visualisasi (tersimpan di `jatim/gambar/`)

Confusion matrix model terbaik, feature importance (jika tree-based), dan tanda tangan spektral rata-rata per kelas.

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if X is None:
    print("STATUS: TUNGGU — visualisasi dilewati.")
else:
    import joblib
    clf = joblib.load(os.path.join(MODEL_DIR, "jatim_best.joblib"))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    cm = confusion_matrix(yte, clf.predict(Xte))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    labs = [NAMES[i] for i in sorted(NAMES)]
    ax.set_xticks(range(6), labs, rotation=20)
    ax.set_yticks(range(6), labs)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=10)
    ax.set_title("Confusion matrix (test set)")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(os.path.join(IMG_DIR, "fig_cm_62.png"), dpi=100)
    plt.close(fig)
    if hasattr(clf, "feature_importances_"):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.barh(FEATS, clf.feature_importances_)
        ax.set_title("Feature importance")
        fig.tight_layout()
        fig.savefig(os.path.join(IMG_DIR, "fig_importance_62.png"), dpi=100)
        plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 5))
    for cid in sorted(NAMES):
        m = df[df.class_id == cid][BANDS].mean().to_numpy()
        ax.plot(BANDS, m, marker="o", label=f"{cid} {NAMES[cid]}")
    ax.set_title("Tanda tangan spektral rata-rata per kelas")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(IMG_DIR, "fig_spektral_62.png"), dpi=100)
    plt.close(fig)
    print("tersimpan: fig_cm_62, fig_importance_62, fig_spektral_62")

tersimpan: fig_cm_62, fig_importance_62, fig_spektral_62


## Status & lanjut

- ▶ **6.3**: peta hasil klasifikasi jendela demo (butuh `jatim/raster/s2_demo_sidoarjo.tif` dari fase `demo`) + Folium Google Satellite.
- ▶ **6.4**: aplikasi Streamlit + deploy.